# Data Loading Pipeline

This notebook walks through each step of the raw data loading process for
the **harxhar** volatility forecasting project. Each cell corresponds to
one stage of the pipeline, with inspection outputs so you can verify
correctness at every point.

The final cell writes the complete `loading.py` module to `../src/loading.py`.

In [ ]:
# ── Setup: clone repo, install deps, set data path ──
import os

REPO_URL = "https://github.com/jamesdchen/harxhar.git"
REPO_DIR = "/content/harxhar"
if not os.path.isdir(REPO_DIR):
    get_ipython().system("git clone {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
get_ipython().system("pip install -q numpy pandas pyarrow")

DATA_PATH = "all30min"

In [ ]:
# ── Parameters ──
# Set to True to keep rows with NaN in feature columns.
# Set to False (default) to drop all rows with any NaN after
# forward-filling the RV target column.
ALLOW_MISSING = False

In [ ]:
# ── Raw parquet inspection ──
import pandas as pd

if os.path.isfile(DATA_PATH):
    parquet_files = [DATA_PATH]
else:
    parquet_files = sorted(f for f in os.listdir(DATA_PATH) if f.endswith(".parquet"))
    parquet_files = [os.path.join(DATA_PATH, f) for f in parquet_files]

print(f"Found {len(parquet_files)} parquet file(s):")
for fp in parquet_files:
    print(f"  {fp}")

frames = []
for fp in parquet_files:
    tmp = pd.read_parquet(fp)
    print(f"\n── {fp} ──")
    print(f"Shape: {tmp.shape}")
    display(tmp.head())
    tmp.info()
    frames.append(tmp)

In [ ]:
# ── Merge: outer join on endbartime ──
from functools import reduce

if len(frames) == 1:
    df = frames[0].copy()
else:
    df = reduce(
        lambda left, right: pd.merge(left, right, on="endbartime", how="outer"),
        frames,
    )

# Rename columns
rename_map = {}
if "endbartime" in df.columns:
    rename_map["endbartime"] = "t"
if "sumret2" in df.columns:
    rename_map["sumret2"] = "RV"
df = df.rename(columns=rename_map)

# Convert t to datetime, drop duplicates
df["t"] = pd.to_datetime(df["t"])
n_before = len(df)
df = df.drop_duplicates(subset="t")
n_after = len(df)

print(f"Merged shape: {df.shape}")
print(f"Duplicates on t removed: {n_before - n_after}")
display(df.head())

In [ ]:
# ── Grid to 30-min frequency ──
START_DATE = "2005-01-01"
FREQ = "30min"

end_date = df["t"].max()
grid = pd.date_range(start=START_DATE, end=end_date, freq=FREQ)
print(f"Grid size: {len(grid)} slots")
print(f"Grid range: {grid[0]} → {grid[-1]}")

df = df.set_index("t").reindex(grid).rename_axis("t").reset_index()

print(f"\nReindexed shape: {df.shape}")
print("\nNaN counts per column:")
print(df.isna().sum())

In [ ]:
# ── Filter market hours ──
FRIDAY_CLOSE = "20:00"
SUNDAY_OPEN = "18:30"

day_of_week = df["t"].dt.dayofweek  # Mon=0 … Sun=6
time_of_day = df["t"].dt.time

friday_close = pd.Timestamp(f"1900-01-01 {FRIDAY_CLOSE}").time()
sunday_open = pd.Timestamp(f"1900-01-01 {SUNDAY_OPEN}").time()

mask_friday_after_close = (day_of_week == 4) & (time_of_day > friday_close)
mask_saturday = day_of_week == 5
mask_sunday_before_open = (day_of_week == 6) & (time_of_day < sunday_open)

closed_mask = mask_friday_after_close | mask_saturday | mask_sunday_before_open

rows_before = len(df)
df = df[~closed_mask].reset_index(drop=True)
rows_after = len(df)

print(f"Rows before filter: {rows_before:,}")
print(f"Rows after filter:  {rows_after:,}")
print(f"Rows removed:       {rows_before - rows_after:,}")

In [ ]:
# ── Forward-fill target column ──
rv_nan_before = df["RV"].isna().sum()
df["RV"] = df["RV"].ffill()
rv_nan_after = df["RV"].isna().sum()

print(f"RV NaNs before ffill: {rv_nan_before:,}")
print(f"RV NaNs after ffill:  {rv_nan_after:,}")

# Drop rows where RV is still NaN (leading NaNs before first observation)
rows_before = len(df)
df = df.dropna(subset=["RV"]).reset_index(drop=True)
rows_after = len(df)

print(f"\nRows dropped (RV still NaN): {rows_before - rows_after:,}")
print("\nRV stats:")
print(df["RV"].describe())

In [ ]:
# ── NaN handling (conditional on ALLOW_MISSING) ──
nan_counts = df.isna().sum()
total_nans = nan_counts.sum()
print(f"Total NaNs remaining: {total_nans:,}")
if total_nans > 0:
    print("\nNaN counts per column:")
    print(nan_counts[nan_counts > 0])

if not ALLOW_MISSING:
    rows_before = len(df)
    df = df.dropna().reset_index(drop=True)
    rows_after = len(df)
    print(f"\nALLOW_MISSING=False → dropped {rows_before - rows_after:,} rows with NaN")
else:
    print("\nALLOW_MISSING=True → keeping rows with NaN in feature columns")

print(f"Final shape: {df.shape}")

In [ ]:
# ── Final inspection ──
df["time_of_day"] = df["t"].dt.time

print(f"Shape: {df.shape}")
print(f"Date range: {df['t'].min()} → {df['t'].max()}")
print(f"Columns: {list(df.columns)}")
print()
display(df.describe())
print()
display(df.head(10))

In [ ]:
%%writefile ../src/loading.py
"""Standalone data loading module for volatility forecasting.

Loads raw parquet data, builds a 30-min grid, filters market hours,
and returns a clean DataFrame ready for downstream pipelines.

No imports from core/ or projects/ — only numpy, pandas, os, functools.
"""

import os
from functools import reduce

import numpy as np
import pandas as pd

# ── Constants ──────────────────────────────────────────────────────────
START_DATE = "2005-01-01"
FRIDAY_CLOSE = "20:00"
SUNDAY_OPEN = "18:30"
FREQ = "30min"


def load_raw_data(data_path: str, allow_missing: bool = False) -> pd.DataFrame:
    """Load parquet data, grid to 30-min, filter market hours, clean NaNs.

    Parameters
    ----------
    data_path : str
        Path to a directory of .parquet files or a single .parquet file.
    allow_missing : bool
        If False (default), drop all rows with any remaining NaN after
        forward-filling the target column. If True, keep them.

    Returns
    -------
    pd.DataFrame
        Cleaned DataFrame with columns including ``t``, ``RV``,
        ``time_of_day``, and any additional feature columns from the
        parquet files.
    """
    # ── 1. Load parquet file(s) ────────────────────────────────────────
    if os.path.isfile(data_path):
        frames = [pd.read_parquet(data_path)]
    else:
        parquet_files = sorted(
            f for f in os.listdir(data_path) if f.endswith(".parquet")
        )
        if not parquet_files:
            raise FileNotFoundError(f"No .parquet files found in {data_path}")
        frames = [
            pd.read_parquet(os.path.join(data_path, f)) for f in parquet_files
        ]

    # ── 2. Merge on endbartime (outer join) ────────────────────────────
    if len(frames) == 1:
        df = frames[0]
    else:
        df = reduce(
            lambda left, right: pd.merge(left, right, on="endbartime", how="outer"),
            frames,
        )

    # ── 3. Rename columns ─────────────────────────────────────────────
    rename_map = {}
    if "endbartime" in df.columns:
        rename_map["endbartime"] = "t"
    if "sumret2" in df.columns:
        rename_map["sumret2"] = "RV"
    df = df.rename(columns=rename_map)

    # ── 4. Convert t to datetime, drop duplicates ─────────────────────
    df["t"] = pd.to_datetime(df["t"])
    df = df.drop_duplicates(subset="t")

    # ── 5. Create 30-min grid and reindex ─────────────────────────────
    end_date = df["t"].max()
    grid = pd.date_range(start=START_DATE, end=end_date, freq=FREQ)
    df = df.set_index("t").reindex(grid).rename_axis("t").reset_index()

    # ── 6. Filter out market-closed hours ─────────────────────────────
    day_of_week = df["t"].dt.dayofweek  # Mon=0 … Sun=6
    time_of_day = df["t"].dt.time

    friday_close = pd.Timestamp(f"1900-01-01 {FRIDAY_CLOSE}").time()
    sunday_open = pd.Timestamp(f"1900-01-01 {SUNDAY_OPEN}").time()

    mask_friday_after_close = (day_of_week == 4) & (time_of_day > friday_close)
    mask_saturday = day_of_week == 5
    mask_sunday_before_open = (day_of_week == 6) & (time_of_day < sunday_open)

    closed_mask = mask_friday_after_close | mask_saturday | mask_sunday_before_open
    df = df[~closed_mask].reset_index(drop=True)

    # ── 7. Forward-fill RV, drop rows where RV is still NaN ──────────
    df["RV"] = df["RV"].ffill()
    df = df.dropna(subset=["RV"]).reset_index(drop=True)

    # ── 8. NaN handling for remaining columns ─────────────────────────
    if not allow_missing:
        df = df.dropna().reset_index(drop=True)

    # ── 9. Add time_of_day column ─────────────────────────────────────
    df["time_of_day"] = df["t"].dt.time

    return df